In [ ]:
import sys; sys.path.append('..')
import inflation, numpy as np, importlib, fd_validation, visualization, parametric_pillows, wall_generation
from numpy.linalg import norm
import MeshFEM, parallelism, benchmark, utils
import periodic_unit_helper
import numpy.linalg as la

In [ ]:
import importlib

In [ ]:
importlib.reload(parametric_pillows)

In [ ]:
h = 5
w = 1
res = 100

triArea = h * w / res
avg_len = triArea ** 0.5

In [ ]:
import igl

In [ ]:
pts, edges = periodic_unit_helper.get_rectangle_wall(h, w, res)

In [ ]:
visualization.plot_line_segments(pts, edges)

In [ ]:
import igl

In [ ]:
scale = 1

In [ ]:
m, marker, n_vx, n_edge = periodic_unit_helper.get_mesh_from_embedded_wall(pts, edges, avg_len, scale, triArea)

In [ ]:
visualization.plot_line_segments(n_vx, n_edge)

In [ ]:
def is_bbox(point):
    if (point[0] == left_x or point[0] == right_x or point[1] == top_y or point[1] == bot_y) and not point_in_box(point, wall_box):
        return True
    return False

In [ ]:
wall_box = get_scaled_box(pts, 1)

In [ ]:
def point_in_box(pt, box):
    box_vx = box[0]
    min_x = np.min(box_vx[:, 0])
    max_x = np.max(box_vx[:, 0])
    min_y = np.min(box_vx[:, 1])
    max_y = np.max(box_vx[:, 1])
    if pt[0] >= min_x and pt[0] <= max_x and pt[1] >= min_y and pt[1] <= max_y:
        return True
    return False

In [ ]:
wall_box

In [ ]:
visualization.plot_2d_mesh(m, pointList=np.where(marker)[0], width=5, height=5)

In [ ]:
ipu = inflation.InflatablePeriodicUnit(m, marker, epsilon = 1e-5)

In [ ]:
import periodic_unit_helper

In [ ]:
fixedVars = periodic_unit_helper.get_center_fixedVars(ipu)

In [ ]:
# isheet.setRelaxedStiffnessEpsilon(1e-6)

In [ ]:
import py_newton_optimizer
opts = py_newton_optimizer.NewtonOptimizerOptions()
opts.useIdentityMetric = True
opts.beta = 1e-4
opts.gradTol = 1e-10

In [ ]:
from tri_mesh_viewer import TriMeshViewer
viewer = TriMeshViewer(ipu, width=768, height=640)
viewer.showWireframe(True)
viewer.show()

In [ ]:
viewer.showWireframe(True)

In [ ]:
ipu.sheet.rigidMotionPinVars

In [ ]:
fd_perturb = np.random.uniform(-1e-3, 1e-3, ipu.numVars())

In [ ]:
# ipu.setVars(ipu.getVars() + fd_perturb)

In [ ]:
import time, vis
benchmark.reset()
ipu.sheet.setUseTensionFieldEnergy(False)
ipu.sheet.setUseHessianProjectedEnergy(True)
ipu.sheet.pressure = 1
opts.niter = 200
framerate = 5 # Update every 5 iterations
def cb(it):
    if it % framerate == 0:
        viewer.update(scalarField=utils.getStrains(ipu.sheet)[:, 0])
cr = inflation.inflation_newton(ipu, fixedVars, opts, callback=cb)
benchmark.report()

In [ ]:
ipu.getVars()[:3]

In [ ]:
import compute_vibrational_modes

In [ ]:
lambdas, modes = compute_vibrational_modes.compute_vibrational_modes(ipu, fixedVars=[], mtype=compute_vibrational_modes.MassMatrixType.FULL, n=16, sigma=-1e-6)


In [ ]:
import mode_viewer, importlib
importlib.reload(mode_viewer);
mview = mode_viewer.ModeViewer(ipu, modes, lambdas, amplitude=0.02 / lambdas[6])
mview.show()